## Jupyter notebook example - Segementation task
### Example using [MendeleyOCT](https://data.mendeley.com/datasets/rscbjbr9sj/2) dataset
**Application**: Using RETFound for Drusen segmentation

**Author**: Yukun Zhou, Salman Shams

**Date**: 08 Jan 2026

**Contribution:**  
This notebook extends the original RETFound classification pipeline to **semantic segmentation** by adding a lightweight decoder on top of the pretrained ViT encoder and training with CE + Dice loss.

**Performance**:

<table align="left">
<tr>
  <th>Dice</th>
  <th>IOU</th>
</tr>
<tr>
  <td>0.4804</td>
  <td>0.3495</td>
</tr>
</table>




## 1. Install environment
1. Follow [RETFound README](https://github.com/rmaphoh/RETFound) to install environment
2. Restart this Jupyter Notebook
3. Select Kernel retfound

> **Note:** Ensure the same PyTorch / timm versions as the original RETFound repository to avoid weight-loading issues.

In [4]:
import sys, torch
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == 'examples': PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print("sys.executable:", sys.executable)
print("torch version:", torch.__version__)

Project root: F:\GitHub\RETFound
sys.executable: f:\GitHub\RETFound\venv3.9\Scripts\python.exe
torch version: 2.8.0+cpu


## 2. Prepare MendeleyOCT dataset
1. Download dataset from the [gdrive](https://drive.google.com/drive/folders/1gBFXrkhRpp8EbTBlTn72h-UvS6a1JYBv?usp=sharing).
2. Put the data folder under the project directory, e.g. "RETFound/MendeleyOCT".

> **Note:**  
The dataset used in this work has been **preprocessed and annotated for the segmentation task**.  
> - Each B-scan was **horizontally cropped from the top and bottom** to focus on the retinal region.  
> - Binary pixel-level annotations were created for **drusen segmentation** (0: background, 1: drusen).  
> - Image–mask pairs are provided in JPEG/PNG format for direct training.
> - Paired format: `images/*.jpeg` ↔ `masks/*_mask.png`

## 3. Hyperparameter and Path Settings
- Backbone: RETFound ViT-Large  
- Task: Binary drusen segmentation  
- Loss: Cross-Entropy + Dice  
- Image size: 256×256  
- Classes: 2 (background, drusen)

> **Note:** Encoder check point can be downloaded from [here](https://drive.google.com/drive/folders/14SQdLuIxfkiqz_zmpvNkd9Ka4NTW3Fml?usp=sharing)

In [5]:
from pathlib import Path

DATA_PATH = Path("MendeleyOCT/Data")
CKPT = Path("checkpoints/checkpoint-best.pth")

IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 20
CE_WEIGHT = "0.1,0.9"

print("Dataset:", DATA_PATH)
print("Checkpoint:", CKPT)


Dataset: MendeleyOCT\Data
Checkpoint: checkpoints\checkpoint-best.pth


## 4. Fine-tuning RETFound for Segmentation

The pretrained ViT encoder is initialized from RETFound weights,  
and a lightweight convolutional decoder is trained for pixel prediction.

In [ ]:
!python main_segmentation.py \
  --data_path MendeleyOCT/Data \
  --epochs 20 \
  --batch_size 4 \
  --finetune RETFound_OCT

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Loaded 826 valid samples from Segmentation/Data/train
Loaded 200 valid samples from Segmentation/Data/val
Loaded 50 valid samples from Segmentation/Data/test
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Position interpolate from 14x14 to 16x16
Pretrained RETFound weights

## 5. Inference and Evaluation

The following script:
- generates overlay visualizations  
- computes Dice and IOU  
- reports final metrics on the test set

In [ ]:
!python inference_segmentation.py \
  --ckpt segmentation_output/best.pth \
  --data_path MendeleyOCT/Data \
  --out_dir segmentation_output/inference


===== Running Inference =====

DRUSEN-100580-1.jpeg       Dice: 0.3590   IoU: 0.2188
DRUSEN-103885-1.jpeg       Dice: 0.5493   IoU: 0.3786
DRUSEN-103885-2.jpeg       Dice: 0.6422   IoU: 0.4730
DRUSEN-103885-3.jpeg       Dice: 0.6422   IoU: 0.4730
DRUSEN-103885-4.jpeg       Dice: 0.6587   IoU: 0.4911
DRUSEN-103885-5.jpeg       Dice: 0.6587   IoU: 0.4911
DRUSEN-142234-1.jpeg       Dice: 0.2874   IoU: 0.1678
DRUSEN-142234-10.jpeg      Dice: 0.3077   IoU: 0.1818
DRUSEN-142234-11.jpeg      Dice: 0.6610   IoU: 0.4937
DRUSEN-142234-12.jpeg      Dice: 0.1434   IoU: 0.0773
DRUSEN-142234-13.jpeg      Dice: 0.5970   IoU: 0.4255
DRUSEN-142234-14.jpeg      Dice: 0.3913   IoU: 0.2432
DRUSEN-142234-15.jpeg      Dice: 0.0000   IoU: 0.0000
DRUSEN-142234-16.jpeg      Dice: 0.0000   IoU: 0.0000
DRUSEN-142234-17.jpeg      Dice: 0.7333   IoU: 0.5789
DRUSEN-142234-18.jpeg      Dice: 0.1111   IoU: 0.0588
DRUSEN-142234-19.jpeg      Dice: 0.0000   IoU: 0.0000
DRUSEN-142234-2.jpeg       Dice: 0.5764   IoU: 0.4

f:\GitHub\RETFound\venv3.9\lib\site-packages\albumentations\__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## Summary

- Extended RETFound to **drusen segmentation** via decoder  
- Reused MAE-pretrained ViT encoder  
- Provided training & inference pipeline  
- Evaluated with Dice and IoU metrics  
- Demonstrated transfer to dense prediction

## Future Work

- Improve annotation consistency  
- Add more diverse OCT data  
- Stronger preprocessing & normalization  
- Deeper decoder (if compute allows)  
- Hyperparameter tuning